In [0]:
DATA_PATH = "/Workspace/Users/aipooja.yes@gmail.com/databricks-ecommerce-data-engineering/data"

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS workspace.bronze.ecommerce_checkpoints;

In [0]:
from pyspark.sql import functions as F

STREAM_PATH = f"{DATA_PATH}/streaming/web_events"

events_schema = """
event_id STRING,
customer_id STRING,
session_id STRING,
event_type STRING,
product_id STRING,
event_timestamp STRING,
device STRING,
source STRING
"""

events_stream = (
    spark.readStream
    .schema(events_schema)
    .option("header", "true")
    .option("pathGlobFilter", "*.csv")
    .csv(STREAM_PATH)
)

events_stream = (
    events_stream
    .withColumn("_processing_time", F.current_timestamp())
)

query = (
    events_stream.writeStream
    .format("delta")
    .outputMode("append")
    .option(
        "checkpointLocation",
        "/Volumes/workspace/bronze/ecommerce_checkpoints/web_events_v2"
    )
    .trigger(availableNow=True)
    .toTable("workspace.bronze.web_events")
)

query.awaitTermination()